# Video Translation (Google Cloud)

Translates a video's audio from English to Hindi using Google Cloud APIs:
**Speech-to-Text** → **Translation** → **Text-to-Speech**.

No GPU or model downloads needed — just an API key.

**Usage:** Set `API_KEY` and `VIDEO_PATH`, then run all cells.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

!pip install -q ffmpeg-python

In [ ]:
# ── Config ──────────────────────────────────────────────────────────
API_KEY = ""  # ← your Google Cloud API key
VIDEO_PATH = "/content/drive/MyDrive/videos/input.mp4"  # input video

# Hindi TTS voice (https://cloud.google.com/text-to-speech/docs/voices)
VOICE_NAME = "hi-IN-Neural2-D"  # Male
# Alternatives: hi-IN-Neural2-A (Female), hi-IN-Neural2-B (Male),
#               hi-IN-Neural2-C (Female), hi-IN-Neural2-D (Male)

assert API_KEY, "Set your Google Cloud API key above"
print(f"Voice: {VOICE_NAME}")

In [ ]:
import os, base64, time, tempfile, wave, json, requests, ffmpeg
from pathlib import Path
from tqdm.auto import tqdm

SAMPLE_RATE = 16_000
TTS_RATE = 24_000
STT_URL = "https://speech.googleapis.com/v1"
TRANSLATE_URL = "https://translation.googleapis.com/language/translate/v2"
TTS_URL = "https://texttospeech.googleapis.com/v1/text:synthesize"


def _api(url, body, params=None):
    """POST to a Google Cloud REST endpoint and return JSON."""
    p = {"key": API_KEY}
    if params:
        p.update(params)
    r = requests.post(url, params=p, json=body)
    if not r.ok:
        msg = r.json().get("error", {}).get("message", r.text)
        raise RuntimeError(f"API error {r.status_code}: {msg}")
    return r.json()


def _wav_duration(path):
    with wave.open(path) as w:
        return w.getnframes() / w.getframerate()


def extract_audio(video_path):
    """Extract 16 kHz mono WAV from video. Returns temp file path."""
    tmp = tempfile.mktemp(suffix=".wav")
    ffmpeg.input(video_path).output(tmp, ac=1, ar=SAMPLE_RATE, format="wav").overwrite_output().run(quiet=True)
    return tmp


def transcribe(wav_path):
    """Transcribe English audio via Google Cloud Speech-to-Text."""
    with open(wav_path, "rb") as f:
        audio_b64 = base64.b64encode(f.read()).decode()
    config = {
        "encoding": "LINEAR16",
        "sampleRateHertz": SAMPLE_RATE,
        "languageCode": "en-US",
        "enableAutomaticPunctuation": True,
    }
    body = {"config": config, "audio": {"content": audio_b64}}
    duration = _wav_duration(wav_path)

    if duration <= 60:
        resp = _api(f"{STT_URL}/speech:recognize", body)
        results = resp.get("results", [])
    else:
        # Long-running recognition for > 60 s
        resp = _api(f"{STT_URL}/speech:longrunningrecognize", body)
        op = resp["name"]
        while True:
            time.sleep(3)
            r = requests.get(f"{STT_URL}/operations/{op}", params={"key": API_KEY})
            poll = r.json()
            if poll.get("done"):
                break
        results = poll.get("response", {}).get("results", [])

    return " ".join(
        r["alternatives"][0]["transcript"]
        for r in results if r.get("alternatives")
    )


def translate(text):
    """Translate English text to Hindi."""
    body = {"q": text, "source": "en", "target": "hi", "format": "text"}
    resp = _api(TRANSLATE_URL, body)
    return resp["data"]["translations"][0]["translatedText"]


def synthesize(text):
    """Synthesize Hindi speech. Returns path to WAV file."""
    # TTS limit is 5000 bytes per request; chunk if needed
    max_chars = 4500
    chunks = [text[i:i + max_chars] for i in range(0, len(text), max_chars)]
    wav_path = tempfile.mktemp(suffix=".wav")

    if len(chunks) == 1:
        body = {
            "input": {"text": chunks[0]},
            "voice": {"languageCode": "hi-IN", "name": VOICE_NAME},
            "audioConfig": {"audioEncoding": "LINEAR16", "sampleRateHertz": TTS_RATE},
        }
        resp = _api(TTS_URL, body)
        with open(wav_path, "wb") as f:
            f.write(base64.b64decode(resp["audioContent"]))
    else:
        with wave.open(wav_path, "wb") as out:
            for i, chunk in enumerate(chunks):
                body = {
                    "input": {"text": chunk},
                    "voice": {"languageCode": "hi-IN", "name": VOICE_NAME},
                    "audioConfig": {"audioEncoding": "LINEAR16", "sampleRateHertz": TTS_RATE},
                }
                resp = _api(TTS_URL, body)
                ptmp = tempfile.mktemp(suffix=".wav")
                with open(ptmp, "wb") as f:
                    f.write(base64.b64decode(resp["audioContent"]))
                with wave.open(ptmp) as w:
                    if i == 0:
                        out.setparams(w.getparams())
                    out.writeframes(w.readframes(w.getnframes()))
                os.remove(ptmp)
    return wav_path


def replace_audio(video_path, audio_wav, output_path):
    """Mux original video with translated audio, adjusting tempo to match."""
    video_dur = float(ffmpeg.probe(video_path)["format"]["duration"])
    audio_dur = _wav_duration(audio_wav)

    vid = ffmpeg.input(video_path)
    aud = ffmpeg.input(audio_wav)

    tempo = audio_dur / video_dur
    if 0.5 <= tempo <= 2.0 and abs(tempo - 1.0) > 0.05:
        aud_stream = aud.audio.filter("atempo", tempo)
    else:
        aud_stream = aud.audio

    ffmpeg.output(
        vid.video, aud_stream, output_path,
        vcodec="copy", acodec="aac", strict="experimental",
    ).overwrite_output().run(quiet=True)


print("Ready.")

In [ ]:
video = Path(VIDEO_PATH)
assert video.exists(), f"Video not found: {VIDEO_PATH}"

output_path = str(video.with_stem(f"{video.stem}_hi"))
print(f"Input:  {VIDEO_PATH}")
print(f"Output: {output_path}\n")

progress = tqdm(total=4, desc="Progress", unit="step")

# 1 — Extract audio
progress.set_postfix_str("extracting audio")
wav_path = extract_audio(VIDEO_PATH)
print(f"Duration: {_wav_duration(wav_path):.1f}s")
progress.update(1)

# 2 — Transcribe
progress.set_postfix_str("transcribing")
transcript = transcribe(wav_path)
os.remove(wav_path)
print(f"Transcript: {transcript[:200]}")
progress.update(1)

# 3 — Translate
progress.set_postfix_str("translating")
hindi_text = translate(transcript)
print(f"Hindi: {hindi_text[:200]}")
progress.update(1)

# 4 — Synthesize & mux
progress.set_postfix_str("synthesizing speech")
tts_wav = synthesize(hindi_text)
replace_audio(VIDEO_PATH, tts_wav, output_path)
os.remove(tts_wav)
progress.update(1)

progress.set_postfix_str("done")
progress.close()
print(f"\nTranslated video saved to:\n{output_path}")